# Step 10 — Robustness terhadap bahasa asli judul dan terjemahan (Tahap 4 dan 19)

**Mengapa perlu:** 46,2% judul (18.266 dari 39.532) berbahasa asli Inggris dan **diterjemahkan ke bahasa Indonesia** (Strategy B, kolom `title_original` menyimpan aslinya). Bahasa asli sangat terkait dengan SINTA (jurnal S1 seluruhnya berbahasa Inggris) dan pangsanya berubah antar-tahun. Karena itu pola topik per-SINTA dan tren per-tahun bisa bercampur dengan efek bahasa/terjemahan. Notebook ini menguji hal itu.

**Input:** `df_lda_topics_v2.csv`, `df_bertopic_topics_v2.csv` (memuat kolom `language` = bahasa asli: `id` atau `en`).

**Yang dilaporkan:**
A. Komposisi bahasa (per SINTA, per tahun, per jurnal).
B. Asosiasi topik-bahasa, dan asosiasi topik-SINTA **di dalam** tiap kelompok bahasa (Cramer's V; tanpa p-value karena artikel dalam satu jurnal tidak independen).
C. Tren: agregat vs di dalam bahasa Indonesia vs di dalam bahasa Inggris vs **standarisasi bahasa** (bobot bahasa tetap). Topik ditandai `language_specific` jika arah slope berlawanan (dan salah satu >= 0,3 pp/tahun) atau jika slope di satu bahasa < 30% dari slope di bahasa lain (dan yang besar >= 0,3 pp/tahun). Ambang ini konvensi, bukan standar baku.

**Catatan untuk Methods:** cantumkan bagaimana terjemahan dibuat (alat/model, versi, tanggal) dan bahwa istilah teknis dilindungi (12 istilah di step3). Kata seperti "menggunakan" kemungkinan sebagian berasal dari terjemahan "using".

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')

# ============ KONFIGURASI ============
LAST_FULL_YEAR = 2024
SLOPE_MIN = 0.3            # pp/tahun
RATIO_MIN = 0.3            # slope stratum lemah harus >= 30% dari stratum kuat
SINTA_GROUPS = {'S1': 'S1-S2', 'S2': 'S1-S2', 'S3': 'S3', 'S4': 'S4-S6', 'S5': 'S4-S6', 'S6': 'S4-S6'}
GROUP_ORDER = ['S1-S2', 'S3', 'S4-S6']
F = dict(lda='df_lda_topics_v2.csv', bt='df_bertopic_topics_v2.csv')
# =====================================
missing = [v for v in F.values() if not os.path.exists(v)]; print('File belum ada:', missing); assert not missing
dl = pd.read_csv(F['lda']); db = pd.read_csv(F['bt'])
assert len(dl) == len(db) and (dl.title_raw.astype(str).values == db.title_raw.astype(str).values).all(), 'Korpus tidak identik'
assert set(dl.language.unique()) <= {'id', 'en'}, 'Nilai language tak terduga'
d = pd.DataFrame({'tahun': dl.tahun, 'jurnal': dl.jurnal, 'sinta': dl.sinta, 'lang': dl.language, 'lda': dl.lda_topic_name, 'bt': db.bertopic_name, 'bt_id': db.bertopic_id_reduced})
d['grp'] = d.sinta.map(SINTA_GROUPS); dB = d[d.bt_id != -1].copy()
print(f'Dokumen = {len(d)} | bahasa asli: {d.lang.value_counts().to_dict()} | BERTopic ber-topik = {len(dB)}')
json.dump(dict(last_full_year=LAST_FULL_YEAR, slope_min_pp=SLOPE_MIN, ratio_min=RATIO_MIN, sinta_groups=SINTA_GROUPS), open('language_config_v2.json', 'w'), indent=2)

## A. Komposisi bahasa asli

In [ ]:
pct_en = lambda s: 100 * (s == 'en').mean()
by_sinta = d.groupby('sinta').lang.agg(n='size', pct_english_original=pct_en).round(1)
by_grp = d.groupby('grp').lang.agg(n='size', pct_english_original=pct_en).loc[GROUP_ORDER].round(1)
by_year = d.groupby('tahun').lang.agg(n='size', pct_english_original=pct_en).round(1)
jl = d.groupby('jurnal').lang.apply(lambda s: (s == 'en').mean())
jstat = pd.Series({'journals_100pct_english': int((jl == 1).sum()), 'journals_0pct_english': int((jl == 0).sum()), 'journals_mixed': int(((jl > 0) & (jl < 1)).sum())})
print(by_sinta.to_string()); print(); print(by_grp.to_string()); print(); print(by_year.to_string()); print(); print(jstat.to_string())
pd.concat({'by_sinta': by_sinta, 'by_group': by_grp, 'by_year': by_year}).to_csv('language_composition_v2.csv')

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
axes[0].bar(by_sinta.index, by_sinta.pct_english_original, color='#4C72B0'); axes[0].set_ylabel('English-original titles (%)'); axes[0].set_xlabel('SINTA'); axes[0].set_title('By SINTA rank')
axes[1].plot(by_year.index, by_year.pct_english_original, '-o', color='#DD8452'); axes[1].axvspan(2024.5, 2025.5, color='grey', alpha=.15); axes[1].set_xlabel('Year'); axes[1].set_title('By year (2025 partial)')
for ax in axes: ax.grid(alpha=.3); ax.set_ylim(0, 105)
fig.suptitle('Figure 15. Share of titles originally in English (machine-translated to Indonesian)', fontsize=10); fig.tight_layout()
plt.savefig('figure15_english_original_share_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## B. Asosiasi topik-bahasa dan topik-SINTA di dalam tiap bahasa
Jika Cramer's V topik-SINTA tetap tinggi **di dalam** satu kelompok bahasa, perbedaan SINTA tidak semata efek terjemahan. Perhatikan ukuran sel: S1-S2 hanya punya 752 artikel berbahasa Indonesia.

In [ ]:
def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct)[0]; n = ct.values.sum(); return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))
res = {}
for nm, df, col in [('LDA', d, 'lda'), ('BERTopic', dB, 'bt')]:
    res[nm] = {'V_topic_x_language': cramers_v(pd.crosstab(df[col], df.lang)), 'V_topic_x_sinta_all': cramers_v(pd.crosstab(df[col], df.grp))}
    for L in ['id', 'en']:
        s = df[df.lang == L]; res[nm][f'V_topic_x_sinta_within_{L}'] = cramers_v(pd.crosstab(s[col], s.grp)); res[nm][f'n_within_{L}'] = int(len(s))
        res[nm][f'group_sizes_within_{L}'] = {k: int(v) for k, v in s.grp.value_counts().items()}
json.dump(res, open('language_association_v2.json', 'w'), indent=2); print(json.dumps(res, indent=2))

for nm, df, col in [('lda', d, 'lda'), ('bertopic', dB, 'bt')]:
    t = df.groupby(col).lang.agg(n_docs='size', pct_english_original=pct_en).round(1).sort_values('pct_english_original', ascending=False)
    t.to_csv(f'topic_language_share_{nm}_v2.csv'); print(f'\n{nm.upper()}: topik dengan porsi judul terjemahan tertinggi/terendah (korpus {pct_en(df.lang):.1f}%)')
    print(pd.concat([t.head(4), t.tail(3)]).to_string())
print('\nOutlier BERTopic (%) menurut bahasa asli:', d.groupby('lang').bt_id.apply(lambda s: round(100 * (s == -1).mean(), 1)).to_dict())

## C. Tren: agregat vs di dalam bahasa vs standarisasi bahasa

In [ ]:
ys = np.arange(2015, LAST_FULL_YEAR + 1)
def lang_trends(df, col):
    df = df[df.tahun <= LAST_FULL_YEAR]; topics = sorted(df[col].unique()); P = {}; S = {}
    for name, sub in [('all', df), ('id', df[df.lang == 'id']), ('en', df[df.lang == 'en'])]:
        c = pd.crosstab(sub.tahun, sub[col]).reindex(index=ys, columns=topics, fill_value=0); P[name] = c.div(c.sum(axis=1), axis=0).fillna(0)
        S[name] = {t: stats.linregress(ys, P[name][t].values) for t in topics}
    w = np.array([(df.lang == 'id').sum(), (df.lang == 'en').sum()], float); w /= w.sum(); std = w[0] * P['id'] + w[1] * P['en']; rows = []
    for t in topics:
        a, b = 100 * S['id'][t].slope, 100 * S['en'][t].slope; hi, lo = max(abs(a), abs(b)), min(abs(a), abs(b)); ss = stats.linregress(ys, std[t].values)
        opposite = (np.sign(a) != np.sign(b)) and hi >= SLOPE_MIN; uneven = (hi >= SLOPE_MIN) and (lo < RATIO_MIN * hi)
        rows.append(dict(Topic=t, slope_all_pp=100 * S['all'][t].slope, p_all=S['all'][t].pvalue, slope_id_pp=a, slope_en_pp=b,
                         slope_lang_standardized_pp=100 * ss.slope, p_lang_standardized=ss.pvalue, same_sign_id_en=bool(np.sign(a) == np.sign(b)),
                         language_specific=bool(opposite or uneven)))
    return pd.DataFrame(rows).sort_values('slope_all_pp', ascending=False).reset_index(drop=True)
TL, TB = lang_trends(d, 'lda'), lang_trends(dB, 'bt')
TL.to_csv('trend_by_language_lda_v2.csv', index=False); TB.to_csv('trend_by_language_bertopic_v2.csv', index=False)
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 52)
cols = ['Topic', 'slope_all_pp', 'slope_id_pp', 'slope_en_pp', 'slope_lang_standardized_pp', 'same_sign_id_en', 'language_specific']
print('LDA:'); print(TL[cols].round(2).to_string(index=False)); print('\nBERTopic:'); print(TB[cols].round(2).to_string(index=False))
for nm, T in [('LDA', TL), ('BERTopic', TB)]:
    print(f"\n{nm}: searah di kedua bahasa {int(T.same_sign_id_en.sum())}/{len(T)} | language_specific {int(T.language_specific.sum())}/{len(T)}")
    print('  ditandai:', '; '.join(f'{r.Topic} (id {r.slope_id_pp:+.2f}, en {r.slope_en_pp:+.2f})' for r in T[T.language_specific].itertuples()))

In [ ]:
flag = pd.concat([TL[TL.language_specific].assign(method='LDA', col='lda'), TB[TB.language_specific].assign(method='BERTopic', col='bt')]).reset_index(drop=True)
n = len(flag); ncols = 4; nrows = int(np.ceil(max(n, 1) / ncols)); fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 2.6 * nrows), sharex=True); axes = np.atleast_1d(axes).ravel()
for ax, r in zip(axes, flag.itertuples()):
    df = d if r.method == 'LDA' else dB; df = df[df.tahun <= LAST_FULL_YEAR]
    for L, c, lab in [('id', '#4C72B0', 'Indonesian-original'), ('en', '#DD8452', 'English-original')]:
        s = df[df.lang == L]; p = pd.crosstab(s.tahun, s[r.col], normalize='index').reindex(index=ys, fill_value=0)[r.Topic]; ax.plot(ys, p, '-o', ms=2.5, lw=1.2, color=c, label=lab)
    ax.set_title(f'{r.method}: {r.Topic[:38]}', fontsize=7); ax.tick_params(labelsize=6); ax.grid(alpha=.25)
for ax in axes[n:]: ax.axis('off')
axes[0].legend(fontsize=6, frameon=False); fig.suptitle('Figure 16. Topics whose trend differs by original language (share within each language, 2015-2024)', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97]); plt.savefig('figure16_language_specific_trends_v2.png', dpi=250, bbox_inches='tight'); plt.show()

## D. File yang dihasilkan
`language_config_v2.json`, `language_composition_v2.csv`, `language_association_v2.json`, `topic_language_share_lda_v2.csv`, `topic_language_share_bertopic_v2.csv`,
`trend_by_language_lda_v2.csv`, `trend_by_language_bertopic_v2.csv`, `figure15_english_original_share_v2.png`, `figure16_language_specific_trends_v2.png`.